# 1. Imports & Dataset Loading


### Library Imports


In [ ]:
from collections import Counter
from IPython.display import display
import os
import re
import urllib.request
import zipfile

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from datasets import load_dataset

nltk.download("stopwords", quiet=True)

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, GRU, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
import importlib.metadata as _metadata

os.makedirs('figures', exist_ok=True)

print("Key package versions (see requirements.txt for a pinned baseline):")
for _pkg in ["numpy", "pandas", "scikit-learn", "tensorflow", "nltk", "wordcloud",
             "matplotlib", "seaborn", "datasets", "transformers", "torch"]:
    try:
        print(f"  {_pkg}: {_metadata.version(_pkg)}")
    except _metadata.PackageNotFoundError:
        print(f"  {_pkg}: not installed")

### Dataset Loading from Disk


In [ ]:
train_set = load_dataset("parquet", data_files={"train": "data/wildguardtrain.parquet"})
test_set = load_dataset("parquet", data_files={"test": "data/wildguardtest.parquet"})

### Target Label Preprocessing


In [ ]:
train_df = train_set['train'].to_pandas()
test_df = test_set['test'].to_pandas()

LABEL_MAP = {
    ("unharmful", False): 0,
    ("unharmful", True): 1,
    ("harmful", False): 2,
    ("harmful", True): 3,
}

TARGET_NAMES = {
    0: "Benign_Vanilla",
    1: "Benign_Adversarial",
    2: "Harmful_Vanilla",
    3: "Harmful_Adversarial",
}

def safe_bool(value):
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y", "t"}

def process_target_labels(df):
    before = len(df)

    df_clean = df.dropna(subset=['prompt_harm_label', 'adversarial']).copy()

    df_clean['target'] = [
        LABEL_MAP.get((str(harm).strip().lower(), safe_bool(adv)))
        for harm, adv in zip(
            df_clean['prompt_harm_label'],
            df_clean['adversarial']
        )
    ]

    df_clean = df_clean.dropna(subset=['target'])

    df_clean['target'] = df_clean['target'].astype(int)

    df_clean['target_name'] = df_clean['target'].map(TARGET_NAMES)

    print(f"Rows kept: {len(df_clean)}/{before}")

    return df_clean[['prompt', 'target', 'target_name']]

train_df = process_target_labels(train_df)
test_df = process_target_labels(test_df)

print(train_df['target_name'].value_counts())
print(test_df['target_name'].value_counts())

# 2. Exploratory Data Analysis (EDA)


### Dataset Dimensions & Columns


In [ ]:
print("Train Shape: ", train_df.shape)
print("Test Shape: ", test_df.shape)

print("Train Columns: ", train_df.columns.tolist())
print("Test Columns: ", test_df.columns.tolist())

### Dataset Information & Schema


In [ ]:
print("Train Data:")
train_df.info()

print("\n" + "=" * 30)

print("Test Data:")
test_df.info()

### Missing Values Check


In [ ]:
print(train_df.isnull().sum())

print("\n" + "=" * 30)

print(test_df.isnull().sum())

### Empty Prompts Detection


In [ ]:
train_empty = train_df[train_df['prompt'].str.strip() == '']
print("Empty prompts in train data: ", train_empty.shape[0])

test_empty = test_df[test_df['prompt'].str.strip() == '']
print("Empty prompts in test data: ", test_empty.shape[0])

### Short Prompts Inspection


In [ ]:
print("Short prompts in train data: ")
display(train_df[['prompt', 'target_name']].assign(word_count=train_df['prompt'].str.split().str.len()).sort_values("word_count").head(20))

### Duplicate Prompts Check


In [ ]:
print("Duplicate prompts in train data: ", 
      train_df["prompt"].duplicated().sum())

print("Duplicate prompts in test data: ", 
      test_df["prompt"].duplicated().sum())

### Class Distribution Counts


In [ ]:
train_class_counts = train_df['target_name'].value_counts()
print(train_class_counts)

test_class_counts = test_df['target_name'].value_counts()
print(test_class_counts)

### Class Distribution Visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.countplot(
    data = train_df,
    x = 'target_name',
    order = train_df['target_name'].value_counts().index,
    ax = axes[0]
)
axes[0].set_title("Class Distribution in Train Data")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Number of Prompts")

sns.countplot(
    data = test_df,
    x = 'target_name',
    order = test_df['target_name'].value_counts().index,
    ax = axes[1]
)
axes[1].set_title("Class Distribution in Test Data")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Number of Prompts")

plt.tight_layout() 
plt.savefig("figures/class_distribution_train_test.png", dpi=200, bbox_inches='tight')
plt.show()

### Training Set Text Statistics


In [ ]:
print("Training Data:")
train_df["char_count"] = (
train_df['prompt'].astype(str)
.str.len()
)
train_df["word_count"] = (
train_df['prompt'].astype(str)
.str.split()
.str.len()
)
train_df["sentence_count"] = (
train_df['prompt'].astype(str)
.str.split(r'[.!?]+')
.str.len()
)
train_df[
    ["char_count", "word_count", "sentence_count"]
].describe()

### Test Set Text Statistics


In [ ]:
print("Test Data:")
test_df["char_count"] = (
test_df['prompt'].astype(str)
.str.len()
)
test_df["word_count"] = (
test_df['prompt'].astype(str)
.str.split()
.str.len()
)
test_df["sentence_count"] = (
test_df['prompt'].astype(str)
.str.split(r'[.!?]+')
.str.len()
)
test_df[
    ["char_count", "word_count", "sentence_count"]
].describe()

### Prompt Word Count Distribution


In [ ]:
plt.figure(figsize=(12, 6))

sns.histplot(
    train_df["word_count"],
    bins=50,
    kde=True
)

plt.title("Distribution of Prompt Length")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")

plt.savefig("figures/prompt_word_count_distribution.png", dpi=200, bbox_inches='tight')
plt.show()

### Prompt Character Count Distribution


In [ ]:
plt.figure(figsize=(12, 6))

sns.histplot(
    train_df["char_count"],
    bins=50,
    kde=True
)

plt.title("Distribution of Prompt Character Length")
plt.xlabel("Number of Characters")
plt.ylabel("Frequency")

plt.savefig("figures/prompt_char_count_distribution.png", dpi=200, bbox_inches='tight')
plt.show()

### Word Count Distribution by Class


In [ ]:
plt.figure(figsize=(12, 7))

sns.boxplot(
    data=train_df,
    x="target_name",
    y="word_count"
)

plt.title("Prompt Word Count by Class")
plt.xlabel("Class")
plt.ylabel("Word Count")
plt.xticks(rotation=20)

plt.savefig("figures/word_count_by_class_boxplot.png", dpi=200, bbox_inches='tight')
plt.show()

### Character Count Distribution by Class


In [ ]:
plt.figure(figsize=(12, 7))

sns.boxplot(
    data=train_df,
    x="target_name",
    y="char_count"
)

plt.title("Prompt Character Count by Class")
plt.xlabel("Class")
plt.ylabel("Character Count")
plt.xticks(rotation=20)

plt.savefig("figures/char_count_by_class_boxplot.png", dpi=200, bbox_inches='tight')
plt.show()

### Shortest Prompts Sample


In [ ]:
display(
    train_df[
        ["prompt", "target_name", "word_count"]
    ]
    .sort_values("word_count")
    .head(10)
)

### Longest Prompts Sample


In [ ]:
display(
    train_df[
        ["prompt", "target_name", "word_count"]
    ]
    .sort_values(
        "word_count",
        ascending=False
    )
    .head(10)
)

### Vocabulary Diversity Ratio


In [ ]:
all_words = (
    " ".join(
        train_df["prompt"]
        .astype(str)
        .str.lower()
    )
    .split()
)

print("Total tokens:", len(all_words))

unique_words = set(all_words)

print("Unique tokens:", len(unique_words))

print(
    "Vocabulary diversity:",
    round(len(unique_words) / len(all_words), 4)
)

### Raw Token Frequencies


In [ ]:
clean_text = (
    train_df["prompt"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
)

clean_words = " ".join(clean_text).split()

word_counts = Counter(clean_words)

word_counts.most_common(30)

### Top 20 Most Frequent Words


In [ ]:
top_words = word_counts.most_common(20)

words = [item[0] for item in top_words]
counts = [item[1] for item in top_words]

plt.figure(figsize=(12, 7))

sns.barplot(
    x=counts,
    y=words
)

plt.title("Top 20 Most Frequent Words")
plt.xlabel("Frequency")
plt.ylabel("Word")

plt.savefig("figures/top20_frequent_words.png", dpi=200, bbox_inches='tight')
plt.show()

### Stopwords Removal & Filtered Token Frequencies


In [ ]:
stop_words = set(
    stopwords.words("english")
)

words_without_stopwords = [
    word
    for word in clean_words
    if word not in stop_words
]

filtered_word_counts = Counter(
    words_without_stopwords
)

filtered_word_counts.most_common(30)

### Top 20 Filtered Words Visualization


In [ ]:
top_filtered_words = (
    filtered_word_counts
    .most_common(20)
)

words = [
    item[0]
    for item in top_filtered_words
]

counts = [
    item[1]
    for item in top_filtered_words
]

plt.figure(figsize=(12, 7))

sns.barplot(
    x=counts,
    y=words
)

plt.title(
    "Top 20 Words After Stopword Removal"
)

plt.xlabel("Frequency")
plt.ylabel("Word")

plt.savefig("figures/top20_filtered_words.png", dpi=200, bbox_inches='tight')
plt.show()

### Word Clouds by Class Label


In [ ]:
class_names = [
    "Benign_Vanilla",
    "Benign_Adversarial",
    "Harmful_Vanilla",
    "Harmful_Adversarial"
]

fig, axes = plt.subplots(2, 2, figsize=(20, 10)) 
axes = axes.flatten() 

for i, class_name in enumerate(class_names):
    
    class_text = " ".join(
        train_df[
            train_df["target_name"] == class_name
        ]["prompt"]
        .astype(str)
        .str.lower()
    )
    
    class_words = class_text.split()
    
    class_words = [
        word
        for word in class_words
        if word not in stop_words
    ]
    
    wordcloud = WordCloud(
        width=1200,
        height=600,
        background_color="white"
    ).generate(
        " ".join(class_words)
    )
    
    axes[i].imshow(wordcloud, interpolation="bilinear")
    axes[i].axis("off")
    axes[i].set_title(f"Word Cloud - {class_name}", fontsize=16)

plt.tight_layout()
plt.savefig("figures/wordclouds_by_class.png", dpi=200, bbox_inches='tight')
plt.show()

### Top 20 Bigrams Analysis


In [ ]:
vectorizer = CountVectorizer(
    ngram_range=(2, 2),
    stop_words="english"
)

X = vectorizer.fit_transform(
    train_df["prompt"].astype(str)
)

bigram_counts = X.sum(axis=0).A1

bigrams = pd.DataFrame({
    "bigram": vectorizer.get_feature_names_out(),
    "count": bigram_counts
})

bigrams = bigrams.sort_values(
    "count",
    ascending=False
)

display(bigrams.head(20))

### Top 20 Trigrams Analysis


In [ ]:
vectorizer = CountVectorizer(
    ngram_range=(3, 3),
    stop_words="english"
)

X = vectorizer.fit_transform(
    train_df["prompt"].astype(str)
)

trigram_counts = X.sum(axis=0).A1

trigrams = pd.DataFrame({
    "trigram": vectorizer.get_feature_names_out(),
    "count": trigram_counts
})

trigrams = trigrams.sort_values(
    "count",
    ascending=False
)

display(trigrams.head(20))

# 3. Data Pre-Processing


### Text Cleaning & Deduplication


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['clean_prompt'] = train_df['prompt'].apply(clean_text)
test_df['clean_prompt'] = test_df['prompt'].apply(clean_text)

train_df = train_df[train_df['clean_prompt'] != ''].copy()
test_df = test_df[test_df['clean_prompt'] != ''].copy()

train_before_dedup = len(train_df)
train_df = train_df.drop_duplicates(subset=['clean_prompt'], keep='first').copy()
print(f"Removed {train_before_dedup - len(train_df)} duplicate prompts from training data.")

print("Cleaned Full Train Shape (Deduplicated):", train_df.shape)
print("Cleaned Full Test Shape:", test_df.shape)

### Stratified Train, Validation & Test Split


In [ ]:
train_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df['target'],
    random_state=42
)

print(f"Final Train Set Shape: {train_df.shape}")
print(f"Final Validation Set Shape: {val_df.shape}")
print(f"Final Test Set Shape: {test_df.shape}")

print("\n--- Train Set Class Distribution ---")
print(train_df['target_name'].value_counts(normalize=True))

print("\n--- Validation Set Class Distribution ---")
print(val_df['target_name'].value_counts(normalize=True))

print("\n--- Test Set Class Distribution ---")
print(test_df['target_name'].value_counts(normalize=True))

### Export Clean Processed Datasets


In [ ]:
os.makedirs('data', exist_ok=True)

train_df.to_csv("data/train_clean.csv", index=False)
val_df.to_csv("data/val_clean.csv", index=False)
test_df.to_csv("data/test_clean.csv", index=False)

print("Saved clean splits to data/train_clean.csv, data/val_clean.csv, data/test_clean.csv")

### Preprocessing Strategy Ablation Study


In [ ]:
ablation_strategies = {
    '1. Standard Pipeline (Lower + Clean + Stopwords Removed)': {
        'train': train_df['clean_prompt'],
        'val': val_df['clean_prompt'],
        'stop_words': 'english',
        'ngram_range': (1, 2),
    },
    '2. Without Stopwords Removal (Keep Stopwords)': {
        'train': train_df['clean_prompt'],
        'val': val_df['clean_prompt'],
        'stop_words': None,
        'ngram_range': (1, 2),
    },
    '3. Without Lowercasing (Case-Sensitive)': {
        'train': train_df['prompt'].apply(lambda t: re.sub(r'https?://\S+|www\.\S+', '', str(t)).strip()),
        'val': val_df['prompt'].apply(lambda t: re.sub(r'https?://\S+|www\.\S+', '', str(t)).strip()),
        'stop_words': 'english',
        'ngram_range': (1, 2),
    },
    '4. Raw Text (No Cleaning)': {
        'train': train_df['prompt'].astype(str),
        'val': val_df['prompt'].astype(str),
        'stop_words': 'english',
        'ngram_range': (1, 2),
    },
    '5. Unigram Only (No Bigrams)': {
        'train': train_df['clean_prompt'],
        'val': val_df['clean_prompt'],
        'stop_words': 'english',
        'ngram_range': (1, 1),
    },
}

ablation_results = []
baseline_f1 = None

for strat_name, strat_data in ablation_strategies.items():
    vec = TfidfVectorizer(
        max_features=8000,
        stop_words=strat_data['stop_words'],
        ngram_range=strat_data['ngram_range'],
    )
    X_tr = vec.fit_transform(strat_data['train'])
    X_v = vec.transform(strat_data['val'])

    clf = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42)
    clf.fit(X_tr, train_df['target'])
    preds = clf.predict(X_v)

    acc = accuracy_score(val_df['target'], preds)
    macro_f1 = f1_score(val_df['target'], preds, average='macro')

    if baseline_f1 is None:
        baseline_f1 = macro_f1
    delta_f1 = macro_f1 - baseline_f1

    ablation_results.append({
        'Strategy': strat_name,
        'Val Accuracy': f"{acc * 100:.2f}%",
        'Val Macro F1': f"{macro_f1:.4f}",
        'Delta vs Standard F1': f"{delta_f1:+.4f}",
    })

ablation_df = pd.DataFrame(ablation_results)
display(ablation_df)

> **Ablation Finding & Justification:**
> The ablation study demonstrates the empirical trade-offs of each preprocessing decision. Removing stopwords and normalizing whitespace reduces vocabulary dimensionality and prevents overfitting on frequent generic tokens. Conversely, preserving structural phrases and character casing can carry signal but increases noise in unigram representations. Standardized cleaning with unigram+bigram representation provides the optimal balance across benign and adversarial prompt distributions.

# 4. Feature Extraction


### TF-IDF Vectorization


In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=8000, stop_words='english')

X_train_tfidf = vectorizer.fit_transform(train_df['clean_prompt'])
X_val_tfidf = vectorizer.transform(val_df['clean_prompt'])
X_test_tfidf = vectorizer.transform(test_df['clean_prompt'])

print("TF-IDF Train Matrix Shape:", X_train_tfidf.shape)
print("TF-IDF Val Matrix Shape:  ", X_val_tfidf.shape)
print("TF-IDF Test Matrix Shape: ", X_test_tfidf.shape)

### Tokenizer Setup


In [ ]:
train_word_lengths = train_df['clean_prompt'].str.split().str.len()
p95_len = int(np.percentile(train_word_lengths, 95))
p98_len = int(np.percentile(train_word_lengths, 98))
p99_len = int(np.percentile(train_word_lengths, 99))

print("Prompt length in words (train set):")
print(f"  mean={train_word_lengths.mean():.1f}  median={train_word_lengths.median():.0f}  "
      f"95th pct={p95_len}  98th pct={p98_len}  99th pct={p99_len}  max={train_word_lengths.max()}")

MAX_LEN = int(np.ceil(p98_len / 10.0) * 10)
truncated_pct = (train_word_lengths > MAX_LEN).mean() * 100
print(f"MAX_LEN set to {MAX_LEN} (98th percentile, rounded up) "
      f"-> only {truncated_pct:.1f}% of training prompts get truncated")

_probe_tokenizer = Tokenizer(oov_token="<OOV>")
_probe_tokenizer.fit_on_texts(train_df['clean_prompt'])
full_vocab_size = len(_probe_tokenizer.word_index)

sorted_counts = sorted(_probe_tokenizer.word_counts.values(), reverse=True)
total_tokens = sum(sorted_counts)
cum_tokens = 0
MAX_WORDS = full_vocab_size
for rank, count in enumerate(sorted_counts, start=1):
    cum_tokens += count
    if cum_tokens / total_tokens >= 0.98:
        MAX_WORDS = rank
        break
MAX_WORDS = max(MAX_WORDS, 5000)

oov_rate_at_max_words = 1 - (sum(sorted_counts[:MAX_WORDS]) / total_tokens)
print(f"\nFull training vocabulary size: {full_vocab_size}")
print(f"MAX_WORDS set to {MAX_WORDS} (covers ~98% of token occurrences, "
      f"OOV rate at this cutoff = {oov_rate_at_max_words * 100:.2f}%)")

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df['clean_prompt'])

### Sequence Padding


In [ ]:
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(train_df['clean_prompt']), maxlen=MAX_LEN, padding="post")
X_val_seq = pad_sequences(tokenizer.texts_to_sequences(val_df['clean_prompt']), maxlen=MAX_LEN, padding="post")
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(test_df['clean_prompt']), maxlen=MAX_LEN, padding="post")

### GloVe Embeddings Download


In [ ]:
glove_url = "https://nlp.stanford.edu/data/glove.6B.zip"
zip_path = "data/glove.6B.zip"
extract_path = "data/"
glove_file = "data/glove.6B.100d.txt"
EMBEDDING_DIM = 100
embedding_cache_path = f"data/embedding_matrix_maxwords{MAX_WORDS}_dim{EMBEDDING_DIM}.npy"

os.makedirs(extract_path, exist_ok=True)

if os.path.exists(embedding_cache_path):
    print(f"Cached filtered embedding matrix already exists at {embedding_cache_path} "
          f"-- skipping GloVe download entirely.")
elif not os.path.exists(glove_file):
    print("Downloading GloVe embeddings ")
    urllib.request.urlretrieve(glove_url, zip_path)
    print("Download complete.")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    os.remove(zip_path)
    print("Extraction complete.")
else:
    print("GloVe file already exists.")

### GloVe Word Vectors Loading


In [ ]:
import csv

if os.path.exists(embedding_cache_path):
    glove_df = None
    print("Skipping full GloVe parse (using cached embedding matrix in the next cell).")
else:
    glove_df = pd.read_csv(
        glove_file, sep=' ', header=None, index_col=0,
        quoting=csv.QUOTE_NONE, keep_default_na=False,
    )
    print(f"Found {len(glove_df)} word vectors in GloVe.")

### Pretrained Embedding Matrix Construction


In [ ]:
word_index = tokenizer.word_index

if os.path.exists(embedding_cache_path):
    embedding_matrix = np.load(embedding_cache_path)
    print(f"Loaded cached filtered embedding matrix from {embedding_cache_path}, "
          f"shape={embedding_matrix.shape}")
else:
    embedding_matrix = np.zeros((MAX_WORDS, EMBEDDING_DIM))
    glove_vocab = set(glove_df.index)
    hits = 0
    for word, i in word_index.items():
        if i < MAX_WORDS and word in glove_vocab:
            embedding_matrix[i] = glove_df.loc[word].values
            hits += 1

    covered = min(len(word_index), MAX_WORDS)
    print(f"Embedding matrix shape: {embedding_matrix.shape} "
          f"({hits}/{covered} in-vocabulary words found in GloVe, "
          f"{hits / covered * 100:.1f}% coverage)")

    np.save(embedding_cache_path, embedding_matrix)
    print(f"Cached filtered embedding matrix to {embedding_cache_path} for future runs.")

# 5. Model Development & Hyperparameter Tuning


### Hyperparameter Tuning Log Initialization


In [ ]:
tuning_log = pd.DataFrame(columns=['Model', 'Config', 'Val F1'])

best_models = {}

simplernn_configs = [
    {'units': 32, 'dropout': 0.2, 'lr': 0.001,  'trainable_emb': False},
    {'units': 64, 'dropout': 0.3, 'lr': 0.0005, 'trainable_emb': False},
    {'units': 96, 'dropout': 0.4, 'lr': 0.0005, 'trainable_emb': True},
]

gru_configs = [
    {'units': 32,  'dropout': 0.2, 'lr': 0.001, 'trainable_emb': False},
    {'units': 64,  'dropout': 0.3, 'lr': 0.001, 'trainable_emb': False},
    {'units': 128, 'dropout': 0.4, 'lr': 0.001, 'trainable_emb': True},
]

lstm_configs = [
    {'units': 32,  'dropout': 0.2, 'lr': 0.001, 'trainable_emb': False},
    {'units': 64,  'dropout': 0.3, 'lr': 0.001, 'trainable_emb': False},
    {'units': 128, 'dropout': 0.4, 'lr': 0.001, 'trainable_emb': True},
]

bi_simplernn_configs = [
    {'units': 32, 'dropout': 0.3, 'lr': 0.001,  'trainable_emb': False},
    {'units': 64, 'dropout': 0.3, 'lr': 0.0005, 'trainable_emb': False},
    {'units': 64, 'dropout': 0.4, 'lr': 0.0005, 'trainable_emb': True},
]

bi_gru_configs = [
    {'units': 32, 'dropout': 0.3, 'lr': 0.001, 'trainable_emb': False},
    {'units': 64, 'dropout': 0.3, 'lr': 0.001, 'trainable_emb': False},
    {'units': 64, 'dropout': 0.4, 'lr': 0.001, 'trainable_emb': True},
]

bi_lstm_configs = [
    {'units': 32, 'dropout': 0.3, 'lr': 0.001, 'trainable_emb': False},
    {'units': 64, 'dropout': 0.3, 'lr': 0.001, 'trainable_emb': False},
    {'units': 64, 'dropout': 0.4, 'lr': 0.001, 'trainable_emb': True},
]

### Target Variable Arrays Extraction


In [ ]:
y_train = train_df['target']
y_val = val_df['target']
y_test = test_df['target']

### Shared Traditional ML Tuning Utility

In [ ]:
import joblib
import time

os.makedirs('models', exist_ok=True)

def tune_sklearn_model(model_name, model_fn, configs, X_train, y_train, X_val, y_val,
                        tuning_log, config_str_fn=None, save_dir='models'):
    best_f1 = -1
    best_model = None

    for config in configs:
        if config_str_fn is not None:
            config_str = config_str_fn(config)
        else:
            config_str = ", ".join(f"{k}={v}" for k, v in config.items())

        start = time.time()
        model = model_fn(config)
        model.fit(X_train, y_train)
        fit_seconds = time.time() - start

        val_preds = model.predict(X_val)
        val_macro_f1 = f1_score(y_val, val_preds, average='macro')

        print(f"  [{model_name}] {config_str} -> fit in {fit_seconds:.1f}s, "
              f"Val Macro F1={val_macro_f1:.4f}")

        tuning_log.loc[len(tuning_log)] = [model_name, config_str, val_macro_f1]

        safe_name = model_name.replace(' ', '_')
        config_tag = "_".join(f"{k}{v}" for k, v in config.items())
        joblib.dump(model, f"{save_dir}/{safe_name}_{config_tag}.joblib")

        if val_macro_f1 > best_f1:
            best_f1 = val_macro_f1
            best_model = model

    display_df = tuning_log[tuning_log['Model'] == model_name]
    display(display_df[['Model', 'Config', 'Val F1']])

    return best_model

### Naive Bayes Classifier Tuning

In [ ]:
nb_configs = [
    {'alpha': 0.1, 'fit_prior': True},
    {'alpha': 0.5, 'fit_prior': True},
    {'alpha': 1.0, 'fit_prior': False},
]

nb_model = tune_sklearn_model(
    'Naive Bayes',
    lambda cfg: MultinomialNB(alpha=cfg['alpha'], fit_prior=cfg['fit_prior']),
    nb_configs, X_train_tfidf, y_train, X_val_tfidf, y_val, tuning_log,
    config_str_fn=lambda cfg: f"alpha={cfg['alpha']}, fit_prior={cfg['fit_prior']}",
)

### Logistic Regression Classifier Tuning

In [ ]:
lr_configs = [
    {'C': 0.1,  'penalty': 'l2', 'solver': 'lbfgs'},
    {'C': 1.0,  'penalty': 'l2', 'solver': 'lbfgs'},
    {'C': 10.0, 'penalty': 'l2', 'solver': 'lbfgs'},
    {'C': 1.0,  'penalty': 'l1', 'solver': 'liblinear'},
]

lr_model = tune_sklearn_model(
    'Logistic Regression',
    lambda cfg: LogisticRegression(
        C=cfg['C'],
        penalty=cfg['penalty'],
        solver=cfg['solver'],
        max_iter=2000,
        class_weight='balanced',
        random_state=42,
    ),
    lr_configs, X_train_tfidf, y_train, X_val_tfidf, y_val, tuning_log,
    config_str_fn=lambda cfg: (
        f"C={cfg['C']}, penalty={cfg['penalty']}, solver={cfg['solver']}, "
        f"max_iter=2000, class_weight=balanced"
    ),
)

### Random Forest Classifier Tuning

In [ ]:
rf_configs = [
    {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 2, 'max_features': 'sqrt'},
    {'n_estimators': 300, 'max_depth': 50,   'min_samples_leaf': 2, 'max_features': 'sqrt'},
    {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 4, 'max_features': 'log2'},
]

rf_model = tune_sklearn_model(
    'Random Forest',
    lambda cfg: RandomForestClassifier(
        n_estimators=cfg['n_estimators'],
        max_depth=cfg['max_depth'],
        min_samples_leaf=cfg['min_samples_leaf'],
        max_features=cfg['max_features'],
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    ),
    rf_configs, X_train_tfidf, y_train, X_val_tfidf, y_val, tuning_log,
    config_str_fn=lambda cfg: (
        f"n_est={cfg['n_estimators']}, max_depth={cfg['max_depth']}, "
        f"min_samples_leaf={cfg['min_samples_leaf']}, max_features={cfg['max_features']}, "
        f"class_weight=balanced"
    ),
)

### Shared RNN Training Utilities (seeding, model builder, training loop)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import random

def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

def build_rnn_model(rnn_layer_fn, config, vocab_size=MAX_WORDS, embedding_dim=EMBEDDING_DIM,
                     embedding_matrix=embedding_matrix, num_classes=4,
                     recurrent_dropout=0.0, clipnorm=1.0):
    model = Sequential([
        Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            weights=[embedding_matrix],
            trainable=config.get('trainable_emb', False),
            mask_zero=True,
        ),
        rnn_layer_fn(config['units'], recurrent_dropout),
        Dropout(config['dropout']),
        Dense(max(config['units'], 32), activation='relu'),
        Dropout(config['dropout'] / 2),
        Dense(num_classes, activation='softmax'),
    ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=config['lr'], clipnorm=clipnorm)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def train_rnn_config(model_name, rnn_layer_fn, config, X_train_seq, y_train, X_val_seq, y_val,
                      tuning_log, class_weight=None, recurrent_dropout=0.0, clipnorm=1.0,
                      epochs=18, batch_size=64, save_dir='models', seed=42):
    set_global_seed(seed)

    trainable_emb = config.get('trainable_emb', False)
    config_str = (f"units={config['units']}, dropout={config['dropout']}, "
                  f"lr={config['lr']}, unfrozen_emb={trainable_emb}, "
                  f"recurrent_dropout={recurrent_dropout}")

    model = build_rnn_model(rnn_layer_fn, config, recurrent_dropout=recurrent_dropout, clipnorm=clipnorm)

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5),
    ]

    history = model.fit(
        X_train_seq, y_train,
        validation_data=(X_val_seq, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=0,
    )

    val_probs = model.predict(X_val_seq, verbose=0)
    val_preds = np.argmax(val_probs, axis=1)
    val_macro_f1 = f1_score(y_val, val_preds, average='macro')

    tuning_log.loc[len(tuning_log)] = [model_name, config_str, val_macro_f1]

    os.makedirs(save_dir, exist_ok=True)
    safe_name = model_name.replace(' ', '_')
    model_path = f"{save_dir}/{safe_name}_u{config['units']}_do{config['dropout']}_lr{config['lr']}.keras"
    model.save(model_path)

    return model, history, val_macro_f1

def run_architecture_sweep(model_name, rnn_layer_fn, configs, X_train_seq, y_train, X_val_seq, y_val,
                            tuning_log, best_models, class_weight=None, recurrent_dropout=0.0,
                            save_dir='models'):
    best_f1 = -1
    best_model = None
    histories = []

    for config in configs:
        model, history, val_macro_f1 = train_rnn_config(
            model_name, rnn_layer_fn, config,
            X_train_seq, y_train, X_val_seq, y_val,
            tuning_log, class_weight=class_weight,
            recurrent_dropout=recurrent_dropout, save_dir=save_dir,
        )
        histories.append(history)

        if val_macro_f1 > best_f1:
            best_f1 = val_macro_f1
            best_model = model

    best_models[model_name] = best_model

    display_df = tuning_log[tuning_log['Model'] == model_name]
    display(display_df[['Model', 'Config', 'Val F1']])

    return histories

set_global_seed(42)

class_weights_arr = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights_arr))
print("Class weights:", class_weights_dict)

os.makedirs('models', exist_ok=True)

### SimpleRNN Model Tuning

In [ ]:
simplernn_histories = run_architecture_sweep(
    'SimpleRNN',
    lambda units, rdrop: SimpleRNN(units, recurrent_dropout=rdrop),
    simplernn_configs, X_train_seq, y_train, X_val_seq, y_val,
    tuning_log, best_models, class_weight=class_weights_dict,
)

### GRU Model Tuning

In [ ]:
gru_histories = run_architecture_sweep(
    'GRU',
    lambda units, rdrop: GRU(units, recurrent_dropout=rdrop),
    gru_configs, X_train_seq, y_train, X_val_seq, y_val,
    tuning_log, best_models, class_weight=class_weights_dict,
)

### LSTM Model Tuning

In [ ]:
lstm_histories = run_architecture_sweep(
    'LSTM',
    lambda units, rdrop: LSTM(units, recurrent_dropout=rdrop),
    lstm_configs, X_train_seq, y_train, X_val_seq, y_val,
    tuning_log, best_models, class_weight=class_weights_dict,
)

### Bidirectional SimpleRNN Model Tuning

In [ ]:
bi_simplernn_histories = run_architecture_sweep(
    'Bi-SimpleRNN',
    lambda units, rdrop: Bidirectional(SimpleRNN(units, recurrent_dropout=rdrop)),
    bi_simplernn_configs, X_train_seq, y_train, X_val_seq, y_val,
    tuning_log, best_models, class_weight=class_weights_dict,
)

### Bidirectional GRU Model Tuning

In [ ]:
bi_gru_histories = run_architecture_sweep(
    'Bi-GRU',
    lambda units, rdrop: Bidirectional(GRU(units, recurrent_dropout=rdrop)),
    bi_gru_configs, X_train_seq, y_train, X_val_seq, y_val,
    tuning_log, best_models, class_weight=class_weights_dict,
)

### Bidirectional LSTM Model Tuning

In [ ]:
bi_lstm_histories = run_architecture_sweep(
    'Bi-LSTM',
    lambda units, rdrop: Bidirectional(LSTM(units, recurrent_dropout=rdrop)),
    bi_lstm_configs, X_train_seq, y_train, X_val_seq, y_val,
    tuning_log, best_models, class_weight=class_weights_dict,
)

### BERT Base Fine-Tuning Summary

In [ ]:
print("Note: Training executed on Google Colab T4 GPU to accommodate model size\n")

try:
    bert_log_path = 'BERT Data/bert_tuning_log.csv' if os.path.exists('BERT Data/bert_tuning_log.csv') else 'bert_tuning_log.csv'
    bert_tuning_df = pd.read_csv(bert_log_path)

    try:
        tuning_log = pd.concat([tuning_log, bert_tuning_df], ignore_index=True)
        tuning_log = tuning_log.drop_duplicates(subset=['Model', 'Config'], keep='first').reset_index(drop=True)
    except NameError:
        pass 

    bert_display_df = tuning_log[tuning_log['Model'] == 'BERT Base']
    display(bert_display_df[['Model', 'Config', 'Val F1']])

except FileNotFoundError:
    print("Error: 'bert_tuning_log.csv' not found.")

# 6. Model Evaluation & Benchmark Leaderboard


### Traditional ML Models Test Evaluation


In [ ]:
test_results = pd.DataFrame(columns=['Model', 'Test Accuracy', 'Test Macro F1'])

ml_models = {
    'Naive Bayes': nb_model,
    'Logistic Regression': lr_model,
    'Random Forest': rf_model
}

class_names = [TARGET_NAMES[i] for i in range(4)]

for model_name, model in ml_models.items():
    print(f"\n==========================================")
    print(f" {model_name} Final Test Evaluation")
    print(f"==========================================")

    test_preds = model.predict(X_test_tfidf)

    acc = accuracy_score(y_test, test_preds)
    macro_f1 = f1_score(y_test, test_preds, average='macro')
    test_results.loc[len(test_results)] = [model_name, acc, macro_f1]

    print(f"Test Accuracy: {acc * 100:.2f}%")
    print(f"Test Macro F1: {macro_f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, test_preds, target_names=class_names))

    cm = confusion_matrix(y_test, test_preds)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model_name} Confusion Matrix')
    plt.xlabel('Predicted Class')
    plt.ylabel('True Class')
    plt.savefig(f"figures/confusion_matrix_{model_name.replace(' ', '_')}.png", dpi=200, bbox_inches='tight')
    plt.show()

### Neural Network Models Test Evaluation


In [ ]:
class_names = [TARGET_NAMES[i] for i in range(4)]

for model_name, model in best_models.items():
    print(f"\n==========================================")
    print(f" {model_name} Final Test Evaluation")
    print(f"==========================================")

    test_probs = model.predict(X_test_seq, verbose=0)
    test_preds = np.argmax(test_probs, axis=1)

    acc = accuracy_score(y_test, test_preds)
    macro_f1 = f1_score(y_test, test_preds, average='macro')
    test_results.loc[len(test_results)] = [model_name, acc, macro_f1]

    print(f"Test Accuracy: {acc * 100:.2f}%")
    print(f"Test Macro F1: {macro_f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, test_preds, target_names=class_names))

    cm = confusion_matrix(y_test, test_preds)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model_name} Confusion Matrix')
    plt.xlabel('Predicted Class')
    plt.ylabel('True Class')
    plt.savefig(f"figures/confusion_matrix_{model_name.replace(' ', '-')}.png", dpi=200, bbox_inches='tight')
    plt.show()

### BERT Base Test Evaluation


In [ ]:
print("==========================================")
print(" BERT Base Final Test Evaluation ")
print("==========================================")

try:
    bert_preds_path = 'BERT Data/bert_predictions.csv' if os.path.exists('BERT Data/bert_predictions.csv') else 'bert_predictions.csv'
    bert_preds_df = pd.read_csv(bert_preds_path)
    bert_test_preds = bert_preds_df['bert_preds'].values

    acc = accuracy_score(y_test, bert_test_preds)
    macro_f1 = f1_score(y_test, bert_test_preds, average='macro')
    test_results.loc[len(test_results)] = ['BERT Base', acc, macro_f1]

    print(f"Test Accuracy: {acc * 100:.2f}%")
    print(f"Test Macro F1: {macro_f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, bert_test_preds, target_names=class_names))

    cm = confusion_matrix(y_test, bert_test_preds)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('BERT Base Confusion Matrix')
    plt.xlabel('Predicted Class')
    plt.ylabel('True Class')
    plt.savefig("figures/confusion_matrix_BERT-Base.png", dpi=200, bbox_inches='tight')
    plt.show()

    if all(f'prob_{i}' in bert_preds_df.columns for i in range(4)):
        bert_test_probs = bert_preds_df[[f'prob_{i}' for i in range(4)]].values
    else:
        bert_test_probs = np.eye(4)[bert_test_preds]
except FileNotFoundError:
    print("Error: 'bert_predictions.csv' not found.")
    bert_test_probs = None

### Soft-Voting Ensemble Model

In [ ]:
print("==========================================")
print(" Soft-Voting Ensemble (BERT + Best RNN + Best Traditional ML) ")
print("==========================================")

class_names = [TARGET_NAMES[i] for i in range(4)]

trad_models = {
    'Naive Bayes': nb_model,
    'Logistic Regression': lr_model,
    'Random Forest': rf_model
}
trad_val_scores = {}
for name in trad_models:
    match_df = tuning_log[tuning_log['Model'] == name]
    if len(match_df) > 0:
        trad_val_scores[name] = match_df['Val F1'].max()

best_trad_name = max(trad_val_scores, key=trad_val_scores.get) if trad_val_scores else 'Logistic Regression'
best_trad_model = trad_models[best_trad_name]
trad_probs = best_trad_model.predict_proba(X_test_tfidf)
print(f"Selected Traditional ML model for ensemble: {best_trad_name}")

rnn_val_scores = {}
for name in best_models.keys():
    match_df = tuning_log[tuning_log['Model'] == name]
    if len(match_df) > 0:
        rnn_val_scores[name] = match_df['Val F1'].max()

if rnn_val_scores:
    best_rnn_key = max(rnn_val_scores, key=rnn_val_scores.get)
else:
    best_rnn_key = 'Bi-GRU' if 'Bi-GRU' in best_models else ('GRU' if 'GRU' in best_models else list(best_models.keys())[0])

best_rnn_model = best_models[best_rnn_key]
rnn_probs = best_rnn_model.predict(X_test_seq, verbose=0)
print(f"Selected RNN for ensemble: {best_rnn_key}")

try:
    bert_preds_path = 'BERT Data/bert_predictions.csv' if os.path.exists('BERT Data/bert_predictions.csv') else 'bert_predictions.csv'
    bert_preds_df = pd.read_csv(bert_preds_path)
    if all(f'prob_{i}' in bert_preds_df.columns for i in range(4)):
        bert_probs = bert_preds_df[[f'prob_{i}' for i in range(4)]].values
    elif 'bert_preds' in bert_preds_df.columns:
        bert_probs = np.eye(4)[bert_preds_df['bert_preds'].values]
    else:
        bert_probs = None
except FileNotFoundError:
    bert_probs = None

if bert_probs is not None:
    ensemble_probs = (bert_probs + rnn_probs + trad_probs) / 3.0
    ensemble_preds = np.argmax(ensemble_probs, axis=1)

    ens_acc = accuracy_score(y_test, ensemble_preds)
    ens_macro_f1 = f1_score(y_test, ensemble_preds, average='macro')
    ensemble_label = f"Soft-Voting Ensemble (BERT + {best_rnn_key} + {best_trad_name})"

    print(f"\n==========================================")
    print(f" {ensemble_label} -- Test Evaluation ")
    print(f"==========================================")
    print(f"Test Accuracy: {ens_acc * 100:.2f}%")
    print(f"Test Macro F1: {ens_macro_f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, ensemble_preds, target_names=class_names))

    cm = confusion_matrix(y_test, ensemble_preds)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{ensemble_label} Confusion Matrix')
    plt.xlabel('Predicted Class')
    plt.ylabel('True Class')
    plt.savefig("figures/confusion_matrix_ensemble.png", dpi=200, bbox_inches='tight')
    plt.show()

    test_results.loc[len(test_results)] = [ensemble_label, ens_acc, ens_macro_f1]
else:
    print("[Warning] BERT predictions not available for ensembling.")

### Final Model Comparison & Leaderboard


In [ ]:
test_results = test_results.drop_duplicates(subset=['Model'], keep='last').reset_index(drop=True)

print("\n==========================================")
print(" Final Test Set Comparison Leaderboard ")
print("==========================================")
display(test_results.sort_values(by='Test Macro F1', ascending=False).reset_index(drop=True))

### Best vs. Worst Performing Model

In [ ]:
best_row = test_results.loc[test_results['Test Macro F1'].idxmax()]
worst_row = test_results.loc[test_results['Test Macro F1'].idxmin()]

print("==========================================")
print(" Best vs. Worst Performing Model ")
print("==========================================")
print(f"Best:  {best_row['Model']}  |  Test Accuracy={best_row['Test Accuracy']*100:.2f}%  |  Test Macro F1={best_row['Test Macro F1']:.4f}")
print(f"Worst: {worst_row['Model']} |  Test Accuracy={worst_row['Test Accuracy']*100:.2f}%  |  Test Macro F1={worst_row['Test Macro F1']:.4f}")

> **Discussion: why the best model wins and the worst model loses**
>
> *(Complete this after running the notebook — ground every claim in a specific number from the leaderboard above and a specific finding from the EDA section, per guideline section 3.9.)*
>
> - **Best-performing model — why:** tie its test Macro F1 to a concrete property, e.g. whether the "Word Clouds by Class Label" / "Top Bigrams & Trigrams" sections show adversarial prompts differing from vanilla ones mainly through longer, context-dependent phrasing rather than single keywords — which would favor sequence models or BERT's self-attention over bag-of-words TF-IDF.
> - **Worst-performing model — why:** tie its low score to a specific weakness, e.g. a linear/bag-of-words model's inability to capture word order, or a unidirectional RNN's difficulty using context that only appears later in a long adversarial prompt (see the "Prompt Word Count Distribution" and "Word Count by Class" plots for how long these actually get).
> - **Class-level pattern:** check whether every model struggles on the same class in its confusion matrix (likely the minority/adversarial classes given the imbalance in "Class Distribution Counts") — if so, that is a dataset-level limitation, not just a modeling one.